In [21]:
import pandas as pd
import csv

file_path = "/Users/jonaslorler/Documents/cordis-HORIZONprojects-csv/project.csv"

# Read the file and fix multi-line records
fixed_rows = []
current_row = []
expected_fields = 0

with open(file_path, 'r', encoding='utf-8') as f:
    # Read header
    header_line = f.readline().strip()
    header = list(csv.reader([header_line], delimiter=';', quotechar='"'))[0]
    expected_fields = len(header)
    
    # Find indices of columns we want
    id_idx = header.index('id')
    title_idx = header.index('title')
    objective_idx = header.index('objective')
    
    # Process data lines
    in_multiline = False
    
    for line_num, line in enumerate(f, start=2):
        # Count fields in current line
        try:
            # Try to parse with csv reader
            parsed_line = list(csv.reader([line.strip()], delimiter=';', quotechar='"'))
            if parsed_line:
                fields = parsed_line[0]
            else:
                fields = []
                
            # Check if this is a complete row
            if len(fields) == expected_fields and not in_multiline:
                # This is a complete row - extract only the columns we want
                fixed_rows.append([fields[id_idx], fields[title_idx], fields[objective_idx]])
            elif len(fields) < expected_fields or in_multiline:
                # This might be a continuation or start of a multi-line field
                if not in_multiline:
                    # Start of a multi-line record
                    current_row = fields
                    in_multiline = True
                else:
                    # Continuation of previous line - append text to objective field (usually the multiline culprit)
                    if current_row and len(current_row) > objective_idx:
                        current_row[objective_idx] += ' ' + line.strip()
                    elif current_row:
                        current_row[-1] += ' ' + line.strip()
                    
                    # Check if we now have enough fields
                    if line.strip().endswith('"'):
                        # Try parsing the combined row
                        combined = ';'.join(current_row)
                        parsed = list(csv.reader([combined], delimiter=';', quotechar='"'))
                        if parsed and len(parsed[0]) >= expected_fields:
                            complete_row = parsed[0][:expected_fields]
                            fixed_rows.append([complete_row[id_idx], complete_row[title_idx], complete_row[objective_idx]])
                            in_multiline = False
                            current_row = []
            else:
                # Row has too many fields - truncate and extract what we need
                fixed_rows.append([fields[id_idx], fields[title_idx], fields[objective_idx]])
                
        except Exception as e:
            # If parsing fails, try to handle it gracefully
            if in_multiline and current_row:
                if len(current_row) > objective_idx:
                    current_row[objective_idx] += ' ' + line.strip()
                else:
                    current_row[-1] += ' ' + line.strip()

# Create DataFrame with just the columns we want
df_complete = pd.DataFrame(fixed_rows, columns=['id', 'title', 'objective'])
print(f"Total rows recovered: {len(df_complete)}")

# Clean up the data
df_complete.columns = df_complete.columns.str.strip('"')

# Convert ID to numeric
df_complete['id'] = pd.to_numeric(df_complete['id'], errors='coerce')

# Remove rows without valid ID
df_complete = df_complete.dropna(subset=['id'])

print(f"Rows after cleanup: {len(df_complete)}")
print(df_complete.info())


# Reset options if needed
pd.reset_option('display.max_colwidth')

# Show first few rows
print("\nFirst 15 rows:")
df_complete.head(15)

Total rows recovered: 15341
Rows after cleanup: 15341
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15341 entries, 0 to 15340
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         15341 non-null  int64 
 1   title      15341 non-null  object
 2   objective  15341 non-null  object
dtypes: int64(1), object(2)
memory usage: 359.7+ KB
None
id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

,id,title,objective
0,101116741,Digitizing Other Economies: A Comparative Appr...,"How do longstanding, primarily non-industrial,..."
1,101163161,MOLECULAR QUANTUM DYNAMICS IN LOW TEMPERATURE ...,The James Webb Space Telescope (JWST) has ushe...
2,101160499,Multiscale modelling of aberrant phase transit...,The spatiotemporal organization of the cell ma...
3,101166905,The first comprehensive Atlas of the Milky Way,The Milky Way is the cosmic environment in whi...
4,101162875,Untapping multiparametric 2D luminescence sens...,Cellular organisms are complex machines whose ...
5,101167314,Making Sense of the Unexpected in the Gravitat...,General Relativity (GR) is more than a century...
6,101165261,Hernando Colón’s universal library. European c...,While newly printed books were circulating wid...
7,101072693,UA EURATOM NCP SUPPORT OF UKRAINIAN RESEARCH E...,UAinEuratom21 is 24 months project is addresse...
8,101172406,FISA 2025 – EURADWASTE’25 conferences on Eura...,"The FISA-EURADWASTE event is a high-level, res..."
9,101114128,Market & Technology Analysis,Adsorbi is a deep tech startup company from Go...


In [27]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import numpy as np

# Prepare the text data - combine title and objective
df_complete['combined_text'] = df_complete['title'] + ' ' + df_complete['objective']

# Remove any rows with missing text
df_complete = df_complete.dropna(subset=['combined_text'])
df_complete = df_complete[df_complete['combined_text'].str.len() > 50]  # Remove very short texts

print(f"Processing {len(df_complete)} documents")

Processing 15341 documents


In [39]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer # Import CountVectorizer
import csv # For saving output with QUOTE_ALL


# --- BERTopic Configuration ---
SAMPLE_SIZE = None  # Set to None to use all data from df_complete
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
# Adjust MIN_TOPIC_SIZE based on your dataset size. If using the small simulation, make it smaller.
# For the full dataset, 15-50 might be reasonable start. For the 10-doc simulation, needs to be smaller e.g. 2-3
MIN_TOPIC_SIZE = 3 # Adjusted for the small 10-doc simulation
NR_TOPICS = "auto"


def run_bertopic_on_dataframe(df: pd.DataFrame):
    """
    Runs BERTopic analysis on a pre-loaded DataFrame with 'id', 'title', 'objective',
    incorporating stopword removal for better topic representations.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("ERROR: Input is not a valid or non-empty DataFrame. Exiting.")
        return None, None # Return None for both results if error

    required_cols = ['id', 'title', 'objective']
    if not all(col in df.columns for col in required_cols):
        print(f"ERROR: DataFrame must contain columns: {required_cols}")
        print(f"Available columns: {df.columns.tolist()}")
        return None, None

    print(f"Processing DataFrame with {len(df)} rows for BERTopic.")

    # --- 1. Combine 'title' and 'objective' ---
    print("\nCombining 'title' and 'objective' columns...")
    df_working = df.copy()
    df_working['title'] = df_working['title'].fillna('').astype(str)
    df_working['objective'] = df_working['objective'].fillna('').astype(str)
    df_working['combined_text'] = df_working['title'] + " " + df_working['objective']
    df_working['combined_text'] = df_working['combined_text'].str.strip()

    print("Successfully created 'combined_text' column.")
    if not df_working.empty:
        example_text_display = df_working['combined_text'].iloc[0][:200] if len(df_working['combined_text'].iloc[0]) > 0 else "N/A (empty)"
        print(f"Example combined text: '{example_text_display}...'")
    
    df_filtered = df_working[df_working['combined_text'] != ''].copy()
    texts_for_bertopic = df_filtered['combined_text'].tolist()
    ids_for_bertopic = df_filtered['id'].tolist()

    if not texts_for_bertopic:
        print("ERROR: No text data available for BERTopic after combining. Check DataFrame content.")
        return None, None
    
    print(f"Number of non-empty documents for BERTopic: {len(texts_for_bertopic)}")

    if len(texts_for_bertopic) < MIN_TOPIC_SIZE * 2 and len(texts_for_bertopic) > 0: # Heuristic: need enough docs for clustering
        print(f"Warning: Number of documents ({len(texts_for_bertopic)}) is very small for min_topic_size={MIN_TOPIC_SIZE}. Results may be poor or errors might occur.")
        # Adjust min_topic_size if too few documents for it
        new_min_topic_size = max(2, len(texts_for_bertopic) // 3) # ensure it's at least 2
        if new_min_topic_size < MIN_TOPIC_SIZE:
            print(f"Adjusting min_topic_size to {new_min_topic_size} due to small dataset size.")
            current_min_topic_size = new_min_topic_size
        else:
            current_min_topic_size = MIN_TOPIC_SIZE
    elif len(texts_for_bertopic) == 0:
        print("ERROR: No documents to process after filtering.")
        return None, None
    else:
        current_min_topic_size = MIN_TOPIC_SIZE


    # Apply sampling if SAMPLE_SIZE is set
    if SAMPLE_SIZE and len(texts_for_bertopic) > SAMPLE_SIZE:
        print(f"Sampling {SAMPLE_SIZE} documents for BERTopic processing...")
        sampled_df_indices = df_filtered.sample(n=SAMPLE_SIZE, random_state=42).index
        texts_to_process = df_filtered.loc[sampled_df_indices, 'combined_text'].tolist()
        ids_to_process = df_filtered.loc[sampled_df_indices, 'id'].tolist()
    else:
        print("Processing all available non-empty documents from the DataFrame...")
        texts_to_process = texts_for_bertopic
        ids_to_process = ids_for_bertopic


    # --- 2. Initialize BERTopic with Stopword Removal ---
    print(f"\nInitializing BERTopic with model: {EMBEDDING_MODEL}")
    print(f"Min topic size: {current_min_topic_size}, Nr topics: {NR_TOPICS}")
    print("Using CountVectorizer with English stopwords for topic representation.")

    # Define a CountVectorizer with English stopwords
    vectorizer = CountVectorizer(stop_words="english")

    topic_model = BERTopic(
        embedding_model=EMBEDDING_MODEL,
        vectorizer_model=vectorizer,  # Use the vectorizer with stopwords
        min_topic_size=current_min_topic_size, # Use potentially adjusted min_topic_size
        nr_topics=NR_TOPICS,
        verbose=True,
    )

    print("Fitting BERTopic model... This can take a significant amount of time.")
    try:
        topics, probabilities = topic_model.fit_transform(texts_to_process)
        print("BERTopic model fitting complete.")
    except Exception as e:
        print(f"ERROR: BERTopic fitting failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

    # --- 3. Explore and Save Results ---
    print("\n--- BERTopic Results ---")
    topic_info = topic_model.get_topic_info()
    print("Top topics found:")
    print(topic_info.head(15))

    if -1 in topic_info['Topic'].values:
        outlier_count_series = topic_info[topic_info['Topic'] == -1]['Count']
        if not outlier_count_series.empty:
            outlier_count = outlier_count_series.iloc[0]
            print(f"\nNumber of outlier documents (Topic -1): {outlier_count}")
        else:
            print("\nNo outlier topic (-1) found in topic_info, or it has no count.")


    results_df = pd.DataFrame({
        'id': ids_to_process,
        'topic_id': topics
    })
    
    topic_labels_map = {topic_id: "_".join([word for word, _ in topic_model.get_topic(topic_id)[:4]]) # Show top 4 words
                        for topic_id in set(topics) if topic_model.get_topic(topic_id) is not None}
    if -1 in set(topics) and -1 not in topic_labels_map : # Handle outliers if not already (e.g. if get_topic(-1) was None)
        topic_labels_map[-1] = "Outliers"
    results_df['topic_label'] = results_df['topic_id'].map(topic_labels_map)
    
    final_output_df_subset = df_filtered[df_filtered['id'].isin(ids_to_process)][['id', 'title', 'objective', 'combined_text']].copy()
    final_output_df = pd.merge(final_output_df_subset, results_df, on='id', how='left')

    output_csv_path = "cordis_projects_with_bertopics_stopwords_removed.csv"
    print(f"\nSaving results with topic assignments to: {output_csv_path}")
    final_output_df.to_csv(output_csv_path, index=False, quoting=csv.QUOTE_ALL)

    model_save_path = "bertopic_cordis_model_stopwords_removed"
    print(f"Saving BERTopic model to: {model_save_path}")
    topic_model.save(model_save_path, serialization="safetensors")

    print("\nScript finished successfully!")
    return final_output_df, topic_model


# --- How to use it ---
if __name__ == "__main__":
    if 'df_complete' in locals() or 'df_complete' in globals():
        print("Using existing df_complete for BERTopic analysis.")
        final_results_df, trained_model = run_bertopic_on_dataframe(df_complete) # Call the function
        if final_results_df is not None:
            print("\nDisplaying head of the final results DataFrame:")
            print(final_results_df.head())
    else:
        print("ERROR: df_complete is not defined. Please ensure it's loaded before running BERTopic analysis.")

2025-05-13 17:05:17,055 - BERTopic - Embedding - Transforming documents to embeddings.


Using existing df_complete for BERTopic analysis.
Processing DataFrame with 15341 rows for BERTopic.

Combining 'title' and 'objective' columns...
Successfully created 'combined_text' column.
Example combined text: 'Digitizing Other Economies: A Comparative Approach How do longstanding, primarily non-industrial, non-capitalist societies adopt and adapt digital technologies in their daily practices and systems of ...'
Number of non-empty documents for BERTopic: 15341
Processing all available non-empty documents from the DataFrame...

Initializing BERTopic with model: all-MiniLM-L6-v2
Min topic size: 3, Nr topics: auto
Using CountVectorizer with English stopwords for topic representation.
Fitting BERTopic model... This can take a significant amount of time.


Batches:   0%|          | 0/480 [00:00<?, ?it/s]

2025-05-13 17:06:24,931 - BERTopic - Embedding - Completed ✓
2025-05-13 17:06:24,932 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-13 17:06:26,413 - BERTopic - Dimensionality - Completed ✓
2025-05-13 17:06:26,414 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-13 17:06:26,739 - BERTopic - Cluster - Completed ✓
2025-05-13 17:06:26,739 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-13 17:06:28,266 - BERTopic - Representation - Completed ✓
2025-05-13 17:06:28,268 - BERTopic - Topic reduction - Reducing number of topics
2025-05-13 17:06:30,347 - BERTopic - Topic reduction - Reduced number of topics from 894 to 615


BERTopic model fitting complete.

--- BERTopic Results ---
Top topics found:
    Topic  Count                                                 Name  \
0      -1   5122                         -1_research_project_new_cell   
1       0    448                     0_climate_food_bioeconomy_forest   
2       1    317                        1_galaxies_dark_stars_stellar   
3       2    278                  2_researchers_night_science_schools   
4       3    262                        3_brain_neural_visual_sensory   
5       4    176                           4_battery_batteries_li_ion   
6       5    154                     5_quantum_qubits_qubit_computers   
7       6    142                6_democracy_democratic_political_news   
8       7    128                          7_marine_ocean_blue_coastal   
9       8    121               8_rehabilitation_patient_spinal_stroke   
10      9    119                9_circular_plastic_plastics_packaging   
11     10    114          10_archaeological_rom

In [53]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import csv

# --- BERTopic Configuration ---
# SAMPLE_SIZE = None # Set to None to use all data from df_complete, or a number for sampling
# Forcing 100 topics might require a decent number of documents.
# If df_complete is small, BERTopic might struggle to find 100 distinct topics.
# Consider SAMPLE_SIZE based on the size of your actual df_complete.
# For now, assuming you want to run on the full df_complete.
SAMPLE_SIZE = None

EMBEDDING_MODEL = "all-mpnet-base-v2"
# Adjust MIN_TOPIC_SIZE based on your dataset size and desired granularity for 100 topics
# A common heuristic: total_docs / nr_topics / (some_factor like 2 or 3)
# If you have 15000 docs and 100 topics, average topic size is 150.
# min_topic_size could be around 25-75.
MIN_TOPIC_SIZE = 25 # You might need to adjust this
NR_TOPICS = 100     # Forcing 100 topics

# Define custom stopwords, including "research" and "project"
CUSTOM_STOPWORDS = ["research", "project", "study", "aims", "objective", "proposal", "horizon", "european", "eu", "article", "paper", "chapter", "section", "conclusion", "introduction", "results", "discussion", "methodology", "approach", "work", "based", "provide", "develop", "development", "understand", "understanding", "data", "analysis", "system", "model", "process", "new", "novel", "high", "low", "within", "also", "however", "therefore", "further", "different", "various", "specific", "general", "important", "key", "main", "focus", "context", "issue", "problem", "solution", "impact", "effect", "role", "field", "area", "use", "application", "potential", "value", "framework"]


def run_bertopic_on_dataframe_v2(df: pd.DataFrame):
    """
    Streamlined BERTopic analysis on a pre-loaded DataFrame.
    - Uses custom stopwords including "research" and "project".
    - Aims for a specified number of topics (e.g., 100).
    - Outputs a cleaner CSV with only id, combined_text, topic_id, topic_label.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("ERROR: Input is not a valid or non-empty DataFrame.")
        return None, None

    required_cols = ['id', 'title', 'objective']
    if not all(col in df.columns for col in required_cols):
        print(f"ERROR: DataFrame must contain columns: {required_cols}. Got: {df.columns.tolist()}")
        return None, None

    print(f"Processing DataFrame with {len(df)} rows for BERTopic.")

    # --- 1. Combine 'title' and 'objective', prepare texts ---
    df_working = df.copy()
    df_working['title'] = df_working['title'].fillna('').astype(str)
    df_working['objective'] = df_working['objective'].fillna('').astype(str)
    df_working['combined_text'] = df_working['title'] + " " + df_working['objective']
    df_working['combined_text'] = df_working['combined_text'].str.strip()
    
    df_filtered = df_working[df_working['combined_text'] != ''].copy()
    texts_for_bertopic = df_filtered['combined_text'].tolist()
    ids_for_bertopic = df_filtered['id'].tolist() # Original IDs

    if not texts_for_bertopic:
        print("ERROR: No text data available for BERTopic after combining.")
        return None, None
    
    print(f"Number of non-empty documents for BERTopic: {len(texts_for_bertopic)}")

    current_min_topic_size = MIN_TOPIC_SIZE
    if len(texts_for_bertopic) < NR_TOPICS * (MIN_TOPIC_SIZE / 2) and len(texts_for_bertopic) > 0 : # Heuristic
         print(f"Warning: Dataset size ({len(texts_for_bertopic)}) might be small for {NR_TOPICS} topics with min_topic_size={MIN_TOPIC_SIZE}.")
         # Optionally adjust min_topic_size if dataset is too small for the target number of topics
         # new_min = max(2, int(len(texts_for_bertopic) / (NR_TOPICS * 1.5) ))
         # if new_min < MIN_TOPIC_SIZE:
         # current_min_topic_size = new_min
         # print(f"Adjusted min_topic_size to {current_min_topic_size}")


    # Apply sampling if SAMPLE_SIZE is set (though it's None by default now)
    if SAMPLE_SIZE and len(texts_for_bertopic) > SAMPLE_SIZE:
        print(f"Sampling {SAMPLE_SIZE} documents for BERTopic processing...")
        sampled_df_indices = df_filtered.sample(n=SAMPLE_SIZE, random_state=42).index
        texts_to_process = df_filtered.loc[sampled_df_indices, 'combined_text'].tolist()
        ids_to_process = df_filtered.loc[sampled_df_indices, 'id'].tolist()
    else:
        texts_to_process = texts_for_bertopic
        ids_to_process = ids_for_bertopic

    if not texts_to_process:
        print("ERROR: No texts to process after sampling (if any).")
        return None, None

    # --- 2. Initialize BERTopic ---
    print(f"\nInitializing BERTopic: Model={EMBEDDING_MODEL}, MinTopicSize={current_min_topic_size}, TargetTopics={NR_TOPICS}")
    
    # Combine sklearn's English stopwords with our custom list
    from sklearn.feature_extraction import _stop_words
    all_stopwords = list(_stop_words.ENGLISH_STOP_WORDS) + CUSTOM_STOPWORDS
    # Remove duplicates that might arise
    all_stopwords = sorted(list(set(all_stopwords)))


    vectorizer = CountVectorizer(stop_words=all_stopwords, ngram_range=(1,1)) # Using (1,1) for now, can try (1,2) later

    topic_model = BERTopic(
        embedding_model=EMBEDDING_MODEL,
        vectorizer_model=vectorizer,
        min_topic_size=current_min_topic_size,
        nr_topics=NR_TOPICS, # Instruct BERTopic to reduce topics to this number
        verbose=True,
        calculate_probabilities=False # Can speed up if probabilities aren't strictly needed for this run
    )

    print("Fitting BERTopic model...")
    try:
        topics, _ = topic_model.fit_transform(texts_to_process) # Probabilities not stored if calculate_probabilities=False
        print("BERTopic model fitting complete.")
    except Exception as e:
        print(f"ERROR: BERTopic fitting failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

    # --- 3. Prepare and Save Results ---
    print("\n--- BERTopic Results ---")
    topic_info = topic_model.get_topic_info()
    print(f"Number of topics found (after potential reduction): {len(topic_info)}")
    print("Top topics found:")
    print(topic_info.head(15)) # Display more topics

    results_df = pd.DataFrame({
        'id': ids_to_process,
        'topic_id': topics
    })
    
    # Generate topic labels (top N words)
    topic_labels_map = {topic_id: "_".join([word for word, _ in topic_model.get_topic(topic_id)[:4]]) # Top 4 words
                        for topic_id in topic_info['Topic'] # Iterate through actual topics found
                        if topic_model.get_topic(topic_id) is not None}
    if -1 in topic_info['Topic'].values and -1 not in topic_labels_map:
        topic_labels_map[-1] = "Outliers"
    results_df['topic_label'] = results_df['topic_id'].map(topic_labels_map)
    
    # Merge with the 'combined_text' from the processed documents
    # df_filtered contains the 'id' and 'combined_text' for all docs that had text.
    # We need to merge results_df (which has 'id' and topic info for *processed* docs) back to this.
    
    # Create a temporary DataFrame with id and combined_text for merging
    processed_texts_df = pd.DataFrame({'id': ids_to_process, 'combined_text': texts_to_process})
    
    # Merge the BERTopic results (topic_id, topic_label) with this processed_texts_df
    final_output_df = pd.merge(processed_texts_df, results_df[['id', 'topic_id', 'topic_label']], on='id', how='left')
    
    # Ensure 'id' column is first, then 'combined_text', 'topic_id', 'topic_label'
    final_output_df = final_output_df[['id', 'combined_text', 'topic_id', 'topic_label']]


    output_csv_path = "cordis_bertopics_100topics_custom_stopwords.csv"
    print(f"\nSaving results to: {output_csv_path}")
    final_output_df.to_csv(output_csv_path, index=False, quoting=csv.QUOTE_ALL)

    model_save_path = "bertopic_cordis_100topics_custom_stopwords_model"
    print(f"Saving BERTopic model to: {model_save_path}")
    topic_model.save(model_save_path, serialization="safetensors")

    print("\nScript finished successfully!")
    return final_output_df, topic_model




In [55]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import csv

# --- BERTopic Configuration ---
SAMPLE_SIZE = None # Set to None to run on the full df_complete

# Use the more powerful all-mpnet-base-v2 model
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# For initial granular topics with combined text (which is richer):
INITIAL_MIN_TOPIC_SIZE = 20 # Can be a bit larger than for titles only (adjust this!)
INITIAL_NR_TOPICS = "auto"  # Let BERTopic find as many as it can

# Target for final general topics:
FINAL_NR_TOPICS = 100

CUSTOM_STOPWORDS = ["research", "project", "study", "aims", "objective", "proposal", "horizon", "european", "eu", "article", "paper", "chapter", "section", "conclusion", "introduction", "results", "discussion", "methodology", "approach", "work", "based", "provide", "develop", "development", "understand", "understanding", "data", "analysis", "system", "model", "process", "new", "novel", "high", "low", "within", "also", "however", "therefore", "further", "different", "various", "specific", "general", "important", "key", "main", "focus", "context", "issue", "problem", "solution", "impact", "effect", "role", "field", "area", "use", "application", "potential", "value", "framework"]


def run_bertopic_combined_text_hierarchical_mpnet(df: pd.DataFrame):
    """
    BERTopic analysis on combined 'title' + 'objective'.
    - Uses sentence-transformers/all-mpnet-base-v2 for embeddings.
    - Uses custom stopwords.
    - Finds granular topics first, then reduces to FINAL_NR_TOPICS.
    - Outputs a CSV with id, combined_text, topic_id, topic_label.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("ERROR: Input is not a valid or non-empty DataFrame.")
        return None, None

    required_cols = ['id', 'title', 'objective']
    if not all(col in df.columns for col in required_cols):
        print(f"ERROR: DataFrame must contain columns: {required_cols}. Got: {df.columns.tolist()}")
        return None, None

    print(f"Processing DataFrame with {len(df)} rows for BERTopic (Combined Text, MPNet, Hierarchical).")

    # --- 1. Combine 'title' and 'objective', prepare texts ---
    df_working = df.copy()
    df_working['title'] = df_working['title'].fillna('').astype(str)
    df_working['objective'] = df_working['objective'].fillna('').astype(str)
    df_working['combined_text'] = df_working['title'] + " " + df_working['objective']
    df_working['combined_text'] = df_working['combined_text'].str.strip()
    
    df_filtered = df_working[df_working['combined_text'] != ''].copy()
    texts_to_process = df_filtered['combined_text'].tolist()
    ids_to_process = df_filtered['id'].tolist() # Original IDs

    if not texts_to_process:
        print("ERROR: No text data available for BERTopic after combining.")
        return None, None
    
    print(f"Number of non-empty combined texts for BERTopic: {len(texts_to_process)}")

    # Apply sampling if SAMPLE_SIZE is set
    if SAMPLE_SIZE and len(texts_to_process) > SAMPLE_SIZE:
        print(f"Sampling {SAMPLE_SIZE} documents for BERTopic processing...")
        # Create a temporary DataFrame for sampling to keep ids and texts aligned
        temp_df_for_sampling = pd.DataFrame({'id': ids_to_process, 'text_data': texts_to_process})
        sampled_data = temp_df_for_sampling.sample(n=SAMPLE_SIZE, random_state=42)
        texts_to_process = sampled_data['text_data'].tolist()
        ids_to_process = sampled_data['id'].tolist()
    # else: # texts_to_process and ids_to_process are already set for full data

    if not texts_to_process:
        print("ERROR: No texts to process after potential sampling.")
        return None, None

    # --- 2. Initialize BERTopic for Granular Topics ---
    print(f"\nStep 1: Initializing BERTopic for GRANULAR topics.")
    print(f"Embedding Model={EMBEDDING_MODEL}, MinTopicSize={INITIAL_MIN_TOPIC_SIZE}, TargetTopics={INITIAL_NR_TOPICS}")
    
    from sklearn.feature_extraction import _stop_words
    all_stopwords = sorted(list(set(list(_stop_words.ENGLISH_STOP_WORDS) + CUSTOM_STOPWORDS)))

    vectorizer = CountVectorizer(stop_words=all_stopwords, ngram_range=(1, 2)) # Using (1,2) ngrams

    topic_model = BERTopic(
        embedding_model=EMBEDDING_MODEL, # Using all-mpnet-base-v2
        vectorizer_model=vectorizer,
        min_topic_size=INITIAL_MIN_TOPIC_SIZE,
        nr_topics=INITIAL_NR_TOPICS, # "auto"
        verbose=True,
        calculate_probabilities=True 
    )

    print("Fitting BERTopic model for GRANULAR topics (using MPNet)... This will take longer.")
    try:
        initial_topics, initial_probabilities = topic_model.fit_transform(texts_to_process)
        print("Initial GRANULAR topic model fitting complete.")
    except Exception as e:
        print(f"ERROR: BERTopic initial fitting failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

    initial_topic_info = topic_model.get_topic_info()
    print(f"\nNumber of initial GRANULAR topics found: {len(initial_topic_info)}")
    print("Top initial granular topics:")
    print(initial_topic_info.head(10))
    
    # --- 3. Reduce to More General Topics ---
    # Only reduce if the number of found topics (excluding outlier) is greater than target
    if (len(initial_topic_info) - (1 if -1 in initial_topic_info["Topic"].values else 0)) > FINAL_NR_TOPICS:
        print(f"\nStep 2: Reducing topics from {len(initial_topic_info)-1} to {FINAL_NR_TOPICS} general topics...")
        try:
            new_topics_assigned = topic_model.reduce_topics(texts_to_process, nr_topics=FINAL_NR_TOPICS)
            print("Topic reduction complete.")
        except Exception as e:
            print(f"ERROR: Topic reduction failed: {e}")
            new_topics_assigned = initial_topics 
            print("Proceeding with initial granular topics due to reduction error.")
    else:
        print(f"\nNumber of initial topics ({len(initial_topic_info)-1}) is already less than or equal to target ({FINAL_NR_TOPICS}). No reduction performed.")
        new_topics_assigned = initial_topics

    # --- 4. Prepare and Save Final Results ---
    final_topic_info = topic_model.get_topic_info()
    print(f"\n--- Final BERTopic Results ({len(final_topic_info)-1 if -1 in final_topic_info['Topic'].values else len(final_topic_info)} General Topics) ---") # Adjusted count
    print("Top general topics found:")
    print(final_topic_info.head(15))

    results_df = pd.DataFrame({
        'id': ids_to_process,
        'topic_id': new_topics_assigned
    })
    
    topic_labels_map = {topic_id: "_".join([word for word, _ in topic_model.get_topic(topic_id)[:4]])
                        for topic_id in final_topic_info['Topic'] 
                        if topic_model.get_topic(topic_id) is not None}
    if -1 in final_topic_info['Topic'].values and -1 not in topic_labels_map:
        topic_labels_map[-1] = "Outliers"
    results_df['topic_label'] = results_df['topic_id'].map(topic_labels_map)
    
    processed_texts_df = pd.DataFrame({'id': ids_to_process, 'combined_text': texts_to_process})
    final_output_df = pd.merge(processed_texts_df, results_df[['id', 'topic_id', 'topic_label']], on='id', how='left')
    final_output_df = final_output_df[['id', 'combined_text', 'topic_id', 'topic_label']]

    output_csv_path = f"cordis_bertopics_mpnet_hierarchical_{FINAL_NR_TOPICS}topics.csv"
    print(f"\nSaving results to: {output_csv_path}")
    final_output_df.to_csv(output_csv_path, index=False, quoting=csv.QUOTE_ALL)

    model_save_path = f"bertopic_cordis_mpnet_hierarchical_{FINAL_NR_TOPICS}topics_model"
    print(f"Saving BERTopic model to: {model_save_path}")
    topic_model.save(model_save_path, serialization="safetensors")

    print("\nScript finished successfully!")
    return final_output_df, topic_model

# --- How to use it ---
if __name__ == "__main__":
    if 'df_complete' not in globals() and 'df_complete' not in locals():
        print("ERROR: df_complete is not defined. Please load or create it first.")
        # For testing, you might want to un-comment a simulation block or load a pickled DataFrame.
        # Example placeholder for testing if df_complete isn't available:
        # data_for_simulation = {'id': [1,2,3]*100, 'title': ['t']*300, 'objective': ['This is a sample objective with several words to ensure text processing works.']*300} # Minimal data
        # df_complete = pd.DataFrame(data_for_simulation)
        # df_complete['id'] = df_complete.index # ensure unique ids for testing
        exit()

    print("Using existing df_complete for HIERARCHICAL BERTopic analysis with MPNet (Combined Text).")
    final_results_df_mpnet, trained_model_mpnet = run_bertopic_combined_text_hierarchical_mpnet(df_complete) # Call the new function
    if final_results_df_mpnet is not None:
        print("\nDisplaying head of the final MPNet hierarchical results DataFrame:")
        print(final_results_df_mpnet.head())
        print(f"\nShape of final results: {final_results_df_mpnet.shape}")
        print(f"\nValue counts for topic_label (top 10):")
        print(final_results_df_mpnet['topic_label'].value_counts().head(10))

2025-05-13 17:48:17,404 - BERTopic - Embedding - Transforming documents to embeddings.


Using existing df_complete for HIERARCHICAL BERTopic analysis with MPNet (Combined Text).
Processing DataFrame with 15341 rows for BERTopic (Combined Text, MPNet, Hierarchical).
Number of non-empty combined texts for BERTopic: 15341

Step 1: Initializing BERTopic for GRANULAR topics.
Embedding Model=sentence-transformers/all-mpnet-base-v2, MinTopicSize=20, TargetTopics=auto
Fitting BERTopic model for GRANULAR topics (using MPNet)... This will take longer.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/480 [00:00<?, ?it/s]

2025-05-13 17:56:23,477 - BERTopic - Embedding - Completed ✓
2025-05-13 17:56:23,477 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-13 17:56:25,321 - BERTopic - Dimensionality - Completed ✓
2025-05-13 17:56:25,321 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-13 17:56:29,636 - BERTopic - Cluster - Completed ✓
2025-05-13 17:56:29,637 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-13 17:56:36,301 - BERTopic - Representation - Completed ✓
2025-05-13 17:56:36,311 - BERTopic - Topic reduction - Reducing number of topics
2025-05-13 17:56:42,853 - BERTopic - Topic reduction - Reduced number of topics from 140 to 50


Initial GRANULAR topic model fitting complete.

Number of initial GRANULAR topics found: 50
Top initial granular topics:
   Topic  Count                                    Name  \
0     -1   5900    -1_technology_social_knowledge_using   
1      0   4284  0_energy_climate_sustainable_solutions   
2      1   1229              1_cell_cells_cancer_immune   
3      2    385      2_health_patients_patient_clinical   
4      3    372        3_quantum_light_states_materials   
5      4    343            4_bone_materials_tissue_soft   
6      5    273       5_galaxies_dark_physics_formation   
7      6    216    6_archaeological_ancient_roman_human   
8      7    145    7_theory_geometry_geometric_problems   
9      8    134            8_brain_neural_memory_visual   

                                                                                               Representation  \
0                          [technology, social, knowledge, using, systems, cell, human, europe, design, time]   
1  

In [58]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import csv
import torch # Import torch to check for MPS availability

# --- BERTopic Configuration ---
SAMPLE_SIZE = None

# Use BAAI/bge-large-en-v1.5
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

INITIAL_MIN_TOPIC_SIZE = 20 # Adjust based on dataset size and text richness
INITIAL_NR_TOPICS = "auto"

FINAL_NR_TOPICS = 100

CUSTOM_STOPWORDS = ["research", "project", "study", "aims", "objective", "proposal", "horizon", "european", "eu", "article", "paper", "chapter", "section", "conclusion", "introduction", "results", "discussion", "methodology", "approach", "work", "based", "provide", "develop", "development", "understand", "understanding", "data", "analysis", "system", "model", "process", "new", "novel", "high", "low", "within", "also", "however", "therefore", "further", "different", "various", "specific", "general", "important", "key", "main", "focus", "context", "issue", "problem", "solution", "impact", "effect", "role", "field", "area", "use", "application", "potential", "value", "framework"]


def run_bertopic_combined_text_hierarchical_bge(df: pd.DataFrame):
    """
    BERTopic analysis on combined 'title' + 'objective'.
    - Uses BAAI/bge-large-en-v1.5 for embeddings.
    - Attempts to use MPS for acceleration on macOS.
    - Uses custom stopwords.
    - Finds granular topics first, then reduces to FINAL_NR_TOPICS.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("ERROR: Input is not a valid or non-empty DataFrame.")
        return None, None

    required_cols = ['id', 'title', 'objective']
    if not all(col in df.columns for col in required_cols):
        print(f"ERROR: DataFrame must contain columns: {required_cols}. Got: {df.columns.tolist()}")
        return None, None

    print(f"Processing DataFrame with {len(df)} rows for BERTopic (Combined Text, BGE-Large, Hierarchical).")

    # --- Check for MPS (Apple Silicon GPU) availability ---
    device = None
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("MPS (Apple Silicon GPU) is available and will be used if sentence-transformers supports it for this model.")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("CUDA GPU is available and will be used.")
    else:
        device = torch.device("cpu")
        print("MPS or CUDA not available, using CPU. This will be slower.")
    
    # Note: SentenceTransformer objects usually auto-detect the best device.
    # If you explicitly pass a device to SentenceTransformer, ensure your PyTorch version fully supports it.
    # For BERTopic, directly passing the model name is often sufficient,
    # and SentenceTransformer (used internally by BERTopic) will handle device placement.

    # --- 1. Combine 'title' and 'objective', prepare texts ---
    df_working = df.copy()
    df_working['title'] = df_working['title'].fillna('').astype(str)
    df_working['objective'] = df_working['objective'].fillna('').astype(str)
    df_working['combined_text'] = df_working['title'] + " " + df_working['objective']
    df_working['combined_text'] = df_working['combined_text'].str.strip()
    
    df_filtered = df_working[df_working['combined_text'] != ''].copy()
    texts_to_process = df_filtered['combined_text'].tolist()
    ids_to_process = df_filtered['id'].tolist()

    if not texts_to_process:
        print("ERROR: No text data available for BERTopic after combining.")
        return None, None
    
    print(f"Number of non-empty combined texts for BERTopic: {len(texts_to_process)}")

    if SAMPLE_SIZE and len(texts_to_process) > SAMPLE_SIZE:
        print(f"Sampling {SAMPLE_SIZE} documents for BERTopic processing...")
        temp_df_for_sampling = pd.DataFrame({'id': ids_to_process, 'text_data': texts_to_process})
        sampled_data = temp_df_for_sampling.sample(n=SAMPLE_SIZE, random_state=42)
        texts_to_process = sampled_data['text_data'].tolist()
        ids_to_process = sampled_data['id'].tolist()

    if not texts_to_process:
        print("ERROR: No texts to process after potential sampling.")
        return None, None

    # --- 2. Initialize BERTopic for Granular Topics ---
    print(f"\nStep 1: Initializing BERTopic for GRANULAR topics.")
    print(f"Embedding Model={EMBEDDING_MODEL}, MinTopicSize={INITIAL_MIN_TOPIC_SIZE}, TargetTopics={INITIAL_NR_TOPICS}")
    
    from sklearn.feature_extraction import _stop_words
    all_stopwords = sorted(list(set(list(_stop_words.ENGLISH_STOP_WORDS) + CUSTOM_STOPWORDS)))

    vectorizer = CountVectorizer(stop_words=all_stopwords, ngram_range=(1, 2))

    # For BGE models, sentence-transformers might require a specific pooling strategy or
    # instructions. Often, just passing the name works, but if not, one might need to
    # create the SentenceTransformer object explicitly.
    # We'll try direct passing first as BERTopic/SentenceTransformer are usually good at this.
    # from sentence_transformers import SentenceTransformer
    # embedding_sbert_model = SentenceTransformer(EMBEDDING_MODEL, device=device if device else None) # Explicit device
    
    topic_model = BERTopic(
        embedding_model=EMBEDDING_MODEL, # Pass BGE model name.
                                        # SentenceTransformer (used by BERTopic) will load it.
                                        # It should auto-detect MPS if available and PyTorch supports it.
        vectorizer_model=vectorizer,
        min_topic_size=INITIAL_MIN_TOPIC_SIZE,
        nr_topics=INITIAL_NR_TOPICS,
        verbose=True,
        calculate_probabilities=True 
    )

    print(f"Fitting BERTopic model for GRANULAR topics (using {EMBEDDING_MODEL})... This will take longer.")
    try:
        initial_topics, initial_probabilities = topic_model.fit_transform(texts_to_process)
        print("Initial GRANULAR topic model fitting complete.")
    except Exception as e:
        print(f"ERROR: BERTopic initial fitting failed: {e}")
        import traceback
        traceback.print_exc()
        if "mps" in str(e).lower() or "metal" in str(e).lower():
            print("\n--- MPS RELATED ERROR DETECTED ---")
            print("Attempting to run on CPU as a fallback...")
            print("Ensure your PyTorch and sentence-transformers versions are compatible with MPS.")
            try:
                # Fallback to CPU if MPS fails
                from sentence_transformers import SentenceTransformer
                cpu_device = torch.device("cpu")
                embedding_sbert_model_cpu = SentenceTransformer(EMBEDDING_MODEL, device=cpu_device)
                topic_model_cpu = BERTopic(
                    embedding_model=embedding_sbert_model_cpu,
                    vectorizer_model=vectorizer,
                    min_topic_size=INITIAL_MIN_TOPIC_SIZE,
                    nr_topics=INITIAL_NR_TOPICS,
                    verbose=True,
                    calculate_probabilities=True
                )
                initial_topics, initial_probabilities = topic_model_cpu.fit_transform(texts_to_process)
                print("Successfully re-ran on CPU after MPS error.")
                topic_model = topic_model_cpu # Replace original model with CPU version
            except Exception as e_cpu:
                print(f"ERROR: Fallback to CPU also failed: {e_cpu}")
                return None, None
        else:
            return None, None # Other non-MPS error

    initial_topic_info = topic_model.get_topic_info()
    print(f"\nNumber of initial GRANULAR topics found: {len(initial_topic_info)}")
    print("Top initial granular topics:")
    print(initial_topic_info.head(10))
    
    # --- 3. Reduce to More General Topics ---
    if (len(initial_topic_info) - (1 if -1 in initial_topic_info["Topic"].values else 0)) > FINAL_NR_TOPICS:
        print(f"\nStep 2: Reducing topics from {len(initial_topic_info)-1} to {FINAL_NR_TOPICS} general topics...")
        try:
            new_topics_assigned = topic_model.reduce_topics(texts_to_process, nr_topics=FINAL_NR_TOPICS)
            print("Topic reduction complete.")
        except Exception as e:
            print(f"ERROR: Topic reduction failed: {e}")
            new_topics_assigned = initial_topics 
            print("Proceeding with initial granular topics due to reduction error.")
    else:
        print(f"\nNumber of initial topics ({len(initial_topic_info)-1}) is already less than or equal to target ({FINAL_NR_TOPICS}). No reduction performed.")
        new_topics_assigned = initial_topics

    # --- 4. Prepare and Save Final Results ---
    final_topic_info = topic_model.get_topic_info()
    final_topic_count = len(final_topic_info) - (1 if -1 in final_topic_info['Topic'].values else 0)
    print(f"\n--- Final BERTopic Results ({final_topic_count} General Topics) ---")
    print("Top general topics found:")
    print(final_topic_info.head(15))

    results_df = pd.DataFrame({
        'id': ids_to_process,
        'topic_id': new_topics_assigned
    })
    
    topic_labels_map = {topic_id: "_".join([word for word, _ in topic_model.get_topic(topic_id)[:4]])
                        for topic_id in final_topic_info['Topic'] 
                        if topic_model.get_topic(topic_id) is not None}
    if -1 in final_topic_info['Topic'].values and -1 not in topic_labels_map:
        topic_labels_map[-1] = "Outliers"
    results_df['topic_label'] = results_df['topic_id'].map(topic_labels_map)
    
    processed_texts_df = pd.DataFrame({'id': ids_to_process, 'combined_text': texts_to_process})
    final_output_df = pd.merge(processed_texts_df, results_df[['id', 'topic_id', 'topic_label']], on='id', how='left')
    final_output_df = final_output_df[['id', 'combined_text', 'topic_id', 'topic_label']]

    output_csv_path = f"cordis_bertopics_bge_large_hierarchical_{FINAL_NR_TOPICS}topics.csv"
    print(f"\nSaving results to: {output_csv_path}")
    final_output_df.to_csv(output_csv_path, index=False, quoting=csv.QUOTE_ALL)

    model_save_path = f"bertopic_cordis_bge_large_hierarchical_{FINAL_NR_TOPICS}topics_model"
    print(f"Saving BERTopic model to: {model_save_path}")
    topic_model.save(model_save_path, serialization="safetensors")

    print("\nScript finished successfully!")
    return final_output_df, topic_model

# --- How to use it ---
if __name__ == "__main__":
    if 'df_complete' not in globals() and 'df_complete' not in locals():
        print("ERROR: df_complete is not defined. Please load or create it first.")
        # Example placeholder for testing if df_complete isn't available:
        # data_for_simulation = {'id': [1,2,3]*100, 'title': ['t']*300, 'objective': ['This is a sample objective for BGE model with several words to ensure text processing works.']*300} # Minimal data
        # df_complete = pd.DataFrame(data_for_simulation)
        # df_complete['id'] = df_complete.index # ensure unique ids for testing
        exit()

    print("Using existing df_complete for HIERARCHICAL BERTopic analysis with BGE-Large (Combined Text).")
    final_results_df_bge, trained_model_bge = run_bertopic_combined_text_hierarchical_bge(df_complete)
    if final_results_df_bge is not None:
        print("\nDisplaying head of the final BGE-Large hierarchical results DataFrame:")
        print(final_results_df_bge.head())
        print(f"\nShape of final results: {final_results_df_bge.shape}")
        print(f"\nValue counts for topic_label (top 10):")
        print(final_results_df_bge['topic_label'].value_counts().head(10))

2025-05-13 18:01:08,489 - BERTopic - Embedding - Transforming documents to embeddings.


Using existing df_complete for HIERARCHICAL BERTopic analysis with BGE-Large (Combined Text).
Processing DataFrame with 15341 rows for BERTopic (Combined Text, BGE-Large, Hierarchical).
MPS (Apple Silicon GPU) is available and will be used if sentence-transformers supports it for this model.
Number of non-empty combined texts for BERTopic: 15341

Step 1: Initializing BERTopic for GRANULAR topics.
Embedding Model=BAAI/bge-large-en-v1.5, MinTopicSize=20, TargetTopics=auto
Fitting BERTopic model for GRANULAR topics (using BAAI/bge-large-en-v1.5)... This will take longer.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/480 [00:00<?, ?it/s]

2025-05-13 18:38:56,671 - BERTopic - Embedding - Completed ✓
2025-05-13 18:38:56,686 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-13 18:38:59,243 - BERTopic - Dimensionality - Completed ✓
2025-05-13 18:38:59,244 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-13 18:39:02,074 - BERTopic - Cluster - Completed ✓
2025-05-13 18:39:02,075 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-13 18:39:09,538 - BERTopic - Representation - Completed ✓
2025-05-13 18:39:09,548 - BERTopic - Topic reduction - Reducing number of topics
2025-05-13 18:39:16,029 - BERTopic - Topic reduction - Reduced number of topics from 109 to 50


Initial GRANULAR topic model fitting complete.

Number of initial GRANULAR topics found: 50
Top initial granular topics:
   Topic  Count                                          Name  \
0     -1   6076        -1_energy_knowledge_technology_systems   
1      0   3756           0_climate_social_sustainable_energy   
2      1   1875                  1_cancer_cell_patients_cells   
3      2    313                2_cell_cells_molecular_protein   
4      3    284            3_galaxies_formation_dark_universe   
5      4    200  4_quantum_computing_quantum computing_qubits   
6      5    178                5_6g_wireless_networks_network   
7      6    176            6_vaccine_clinical_vaccines_health   
8      7    147          7_theory_geometry_geometric_problems   
9      8    144                    8_traffic_road_rail_safety   

                                                                                                             Representation  \
0                          [energy, 

In [60]:
final_results_df_bge.head(100)

,id,combined_text,topic_id,topic_label
0,101116741,"Digitizing Other Economies: A Comparative Approach How do longstanding, primarily non-industrial, non-capitalist societies adopt and adapt digital technologies in their daily practices and systems of values? Classical anthropological theory once arranged basic economic types on an evolutionary ladder ranging from hunter-gatherers, horticulturalists, pastoralists, and agriculturalists to industrialists. Today, the existence of these economies other than industrialism are correctly approached not as anachronisms but as contemporaneous to (post-)industrial life. Still, research on digitization has largely taken place in (post-) industrial contexts, meaning we know next to nothing about how different types of longstanding economies adopt and adapt digital technologies. At the same time, researchers have stipulated that digitization threatens global economic diversity. By comparing digitization to processes of colonization, they have argued that digital technologies facilitate assimilation into (post-)industrial economic systems and their often capitalist values by virtue of their technological design. This project empirically investigates these claims through in-depth ethnographic research among hunter-gatherers (Brazilian Amazon), pastoralists (Kyrgyz Republic), horticulturalists (Solomon Islands) and indigenous agriculturalists (India) who have long resisted assimilation into industrial-capitalism. Additional ethnological comparison of the four sites will offer unique macro-level insights into the possibilities for economic diversity in the digital age. Finally, the project advances a novel theoretical and methodological approach that advances both ethnographic research and ethnological comparison. This approach recognizes the significance of both technological design and contextual adaptations and provides tools for new research agendas not just on digital industrial-capitalism but on diverse economic systems and values.",-1,energy_knowledge_technology_systems
1,101163161,"MOLECULAR QUANTUM DYNAMICS IN LOW TEMPERATURE CONDENSED PHASE ASTROCHEMISTRY The James Webb Space Telescope (JWST) has ushered in a new era in observational astrochemistry. JWST's ability to obtain infrared spectra of molecular ices condensed on interstellar dust grains in dense, star-forming clouds and in protoplanetary disks is expected to revolutionize the field, since these ices are known to be important sources of complex organic molecules. On a fundamental level, the physico-chemical behavior of these ices obeys the laws of molecular quantum dynamics occurring in low-temperature condensed phases. This is a forefront research area in chemical physics that, unfortunately, remains poorly understood. This project establishes an interdisciplinary, synergistic research consortium to address this knowledge deficit, bringing together the unique expertise of the groups of Alec Wodtke (Chemical Physics at Surfaces), Liv Hornekr (Astrochemistry and Scanning Tunneling Microscopy) and Peter Saalfrank (Theoretical Quantum Dynamics). Under IRASTRO, we will develop and employ advanced infrared technology based on superconducting nanowire single-photon detectors (SNSPDs) for new experimental capability in laboratory experiments directly relevant to astrochemistry. We will combine megapixel SNSPD arrays with chelle spectrometers enabling solid-state mid-infrared emission spectroscopy, including single-molecule mid-IR spectroscopy in a scanning tunneling microscope. With these new experiments and forefront quantum theory, we will tackle three research themes: 1) Infrared Spectra of Molecules on Surfaces under Interstellar Conditions, 2) Energy Dissipation Channels on Low Temperature Surfaces, and 3) Chemical Reactivity under Interstellar Conditions. IRASTROs focus on IR spectroscopy will make the projects findings directly relevant to the interpretation of JWST observational data and, through a fruitful collaboration of 

In [57]:
# Assuming df_complete is your DataFrame with id, title, objective
final_results_df_mpnet.head(20)

,id,combined_text,topic_id,topic_label
0,101116741,"Digitizing Other Economies: A Comparative Approach How do longstanding, primarily non-industrial, non-capitalist societies adopt and adapt digital technologies in their daily practices and systems of values? Classical anthropological theory once arranged basic economic types on an evolutionary ladder ranging from hunter-gatherers, horticulturalists, pastoralists, and agriculturalists to industrialists. Today, the existence of these economies other than industrialism are correctly approached not as anachronisms but as contemporaneous to (post-)industrial life. Still, research on digitization has largely taken place in (post-) industrial contexts, meaning we know next to nothing about how different types of longstanding economies adopt and adapt digital technologies. At the same time, researchers have stipulated that digitization threatens global economic diversity. By comparing digitization to processes of colonization, they have argued that digital technologies facilitate assimilation into (post-)industrial economic systems and their often capitalist values by virtue of their technological design. This project empirically investigates these claims through in-depth ethnographic research among hunter-gatherers (Brazilian Amazon), pastoralists (Kyrgyz Republic), horticulturalists (Solomon Islands) and indigenous agriculturalists (India) who have long resisted assimilation into industrial-capitalism. Additional ethnological comparison of the four sites will offer unique macro-level insights into the possibilities for economic diversity in the digital age. Finally, the project advances a novel theoretical and methodological approach that advances both ethnographic research and ethnological comparison. This approach recognizes the significance of both technological design and contextual adaptations and provides tools for new research agendas not just on digital industrial-capitalism but on diverse economic systems and values.",-1,technology_social_knowledge_using
1,101163161,"MOLECULAR QUANTUM DYNAMICS IN LOW TEMPERATURE CONDENSED PHASE ASTROCHEMISTRY The James Webb Space Telescope (JWST) has ushered in a new era in observational astrochemistry. JWST's ability to obtain infrared spectra of molecular ices condensed on interstellar dust grains in dense, star-forming clouds and in protoplanetary disks is expected to revolutionize the field, since these ices are known to be important sources of complex organic molecules. On a fundamental level, the physico-chemical behavior of these ices obeys the laws of molecular quantum dynamics occurring in low-temperature condensed phases. This is a forefront research area in chemical physics that, unfortunately, remains poorly understood. This project establishes an interdisciplinary, synergistic research consortium to address this knowledge deficit, bringing together the unique expertise of the groups of Alec Wodtke (Chemical Physics at Surfaces), Liv Hornekr (Astrochemistry and Scanning Tunneling Microscopy) and Peter Saalfrank (Theoretical Quantum Dynamics). Under IRASTRO, we will develop and employ advanced infrared technology based on superconducting nanowire single-photon detectors (SNSPDs) for new experimental capability in laboratory experiments directly relevant to astrochemistry. We will combine megapixel SNSPD arrays with chelle spectrometers enabling solid-state mid-infrared emission spectroscopy, including single-molecule mid-IR spectroscopy in a scanning tunneling microscope. With these new experiments and forefront quantum theory, we will tackle three research themes: 1) Infrared Spectra of Molecules on Surfaces under Interstellar Conditions, 2) Energy Dissipation Channels on Low Temperature Surfaces, and 3) Chemical Reactivity under Interstellar Conditions. IRASTROs focus on IR spectroscopy will make the projects findings directly relevant to the interpretation of JWST observational data and, through a fruitful collaboration of ex

In [51]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import csv

# --- BERTopic Configuration ---
SAMPLE_SIZE = None
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
MIN_TOPIC_SIZE = 50 # Titles are shorter, might need smaller min_topic_size for 100 topics
NR_TOPICS = 100

CUSTOM_STOPWORDS = ["research", "project", "study", "aims", "objective", "proposal", "horizon", "european", "eu", "article", "paper", "chapter", "section", "conclusion", "introduction", "results", "discussion", "methodology", "approach", "work", "based", "provide", "develop", "development", "understand", "understanding", "data", "analysis", "system", "model", "process", "new", "novel", "high", "low", "within", "also", "however", "therefore", "further", "different", "various", "specific", "general", "important", "key", "main", "focus", "context", "issue", "problem", "solution", "impact", "effect", "role", "field", "area", "use", "application", "potential", "value", "framework"]


def run_bertopic_on_titles(df: pd.DataFrame):
    """
    BERTopic analysis on project titles only.
    - Uses custom stopwords.
    - Aims for a specified number of topics (e.g., 100).
    - Outputs a CSV with id, title, topic_id, topic_label.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("ERROR: Input is not a valid or non-empty DataFrame.")
        return None, None

    required_cols = ['id', 'title'] # Now only 'id' and 'title' are strictly required from input
    if not all(col in df.columns for col in required_cols):
        print(f"ERROR: DataFrame must contain columns: {required_cols}. Got: {df.columns.tolist()}")
        return None, None

    print(f"Processing DataFrame with {len(df)} rows for BERTopic (using titles only).")

    # --- 1. Prepare texts (titles only) ---
    df_working = df.copy()
    df_working['title'] = df_working['title'].fillna('').astype(str).str.strip()
    
    df_filtered = df_working[df_working['title'] != ''].copy() # Filter out rows with empty titles
    texts_for_bertopic = df_filtered['title'].tolist()
    ids_for_bertopic = df_filtered['id'].tolist() # Original IDs

    if not texts_for_bertopic:
        print("ERROR: No title data available for BERTopic.")
        return None, None
    
    print(f"Number of non-empty titles for BERTopic: {len(texts_for_bertopic)}")

    current_min_topic_size = MIN_TOPIC_SIZE
    if len(texts_for_bertopic) < NR_TOPICS * (MIN_TOPIC_SIZE / 2) and len(texts_for_bertopic) > 0 :
         print(f"Warning: Dataset size ({len(texts_for_bertopic)}) might be small for {NR_TOPICS} topics with min_topic_size={MIN_TOPIC_SIZE}.")


    # Apply sampling if SAMPLE_SIZE is set
    if SAMPLE_SIZE and len(texts_for_bertopic) > SAMPLE_SIZE:
        print(f"Sampling {SAMPLE_SIZE} titles for BERTopic processing...")
        # Create a temporary DataFrame for sampling to keep ids and texts aligned
        temp_df_for_sampling = pd.DataFrame({'id': ids_for_bertopic, 'title_text': texts_for_bertopic})
        sampled_data = temp_df_for_sampling.sample(n=SAMPLE_SIZE, random_state=42)
        texts_to_process = sampled_data['title_text'].tolist()
        ids_to_process = sampled_data['id'].tolist()
    else:
        texts_to_process = texts_for_bertopic
        ids_to_process = ids_for_bertopic

    if not texts_to_process:
        print("ERROR: No titles to process after sampling (if any).")
        return None, None

    # --- 2. Initialize BERTopic ---
    print(f"\nInitializing BERTopic: Model={EMBEDDING_MODEL}, MinTopicSize={current_min_topic_size}, TargetTopics={NR_TOPICS}")
    
    from sklearn.feature_extraction import _stop_words
    all_stopwords = sorted(list(set(list(_stop_words.ENGLISH_STOP_WORDS) + CUSTOM_STOPWORDS)))

    vectorizer = CountVectorizer(stop_words=all_stopwords, ngram_range=(1,1))

    topic_model = BERTopic(
        embedding_model=EMBEDDING_MODEL,
        vectorizer_model=vectorizer,
        min_topic_size=current_min_topic_size,
        nr_topics=NR_TOPICS,
        verbose=True,
        calculate_probabilities=False
    )

    print("Fitting BERTopic model on titles...")
    try:
        topics, _ = topic_model.fit_transform(texts_to_process)
        print("BERTopic model fitting complete.")
    except Exception as e:
        print(f"ERROR: BERTopic fitting failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

    # --- 3. Prepare and Save Results ---
    print("\n--- BERTopic Results (Titles Only) ---")
    topic_info = topic_model.get_topic_info()
    print(f"Number of topics found (after potential reduction): {len(topic_info)}")
    print("Top topics found:")
    print(topic_info.head(15))

    # DataFrame with original IDs and assigned topic_ids
    results_df = pd.DataFrame({
        'id': ids_to_process, 
        'topic_id': topics
    })
    
    topic_labels_map = {topic_id: "_".join([word for word, _ in topic_model.get_topic(topic_id)[:4]])
                        for topic_id in topic_info['Topic'] 
                        if topic_model.get_topic(topic_id) is not None}
    if -1 in topic_info['Topic'].values and -1 not in topic_labels_map:
        topic_labels_map[-1] = "Outliers"
    results_df['topic_label'] = results_df['topic_id'].map(topic_labels_map)
    
    # Merge with the 'title' from the processed documents for the final output
    # df_filtered contains the 'id' and 'title' for all docs that had non-empty titles.
    
    # Create a temporary DataFrame with id and title for merging
    # This ensures we only include the titles that were actually processed
    processed_titles_df = pd.DataFrame({'id': ids_to_process, 'title': texts_to_process})
    
    # Merge the BERTopic results (topic_id, topic_label) with this processed_titles_df
    final_output_df = pd.merge(processed_titles_df, results_df[['id', 'topic_id', 'topic_label']], on='id', how='left')
    
    # Ensure 'id' column is first, then 'title', 'topic_id', 'topic_label'
    final_output_df = final_output_df[['id', 'title', 'topic_id', 'topic_label']]

    output_csv_path = "cordis_bertopics_titles_100topics.csv"
    print(f"\nSaving results to: {output_csv_path}")
    final_output_df.to_csv(output_csv_path, index=False, quoting=csv.QUOTE_ALL)

    model_save_path = "bertopic_cordis_titles_100topics_model"
    print(f"Saving BERTopic model to: {model_save_path}")
    topic_model.save(model_save_path, serialization="safetensors")

    print("\nScript finished successfully!")
    return final_output_df, topic_model

# --- How to use it ---
if __name__ == "__main__":
    if 'df_complete' not in globals() and 'df_complete' not in locals():
        print("ERROR: df_complete is not defined. Please load or create it first.")
        # Example placeholder for testing if df_complete isn't available:
        # data_for_simulation = {'id': [1,2,3]*100, 'title': ['Example Title Word']*300, 'objective': ['o']*300} # Minimal data
        # df_complete = pd.DataFrame(data_for_simulation)
        # df_complete['id'] = df_complete.index # ensure unique ids for testing
        exit()

    print("Using existing df_complete for BERTopic analysis (titles only).")
    final_results_df_titles, trained_model_titles = run_bertopic_on_titles(df_complete)
    if final_results_df_titles is not None:
        print("\nDisplaying head of the final results DataFrame (titles only):")
        print(final_results_df_titles.head())
        print(f"\nShape of final results: {final_results_df_titles.shape}")
        print(f"\nValue counts for topic_label (top 10):")
        print(final_results_df_titles['topic_label'].value_counts().head(10))

2025-05-13 17:35:43,266 - BERTopic - Embedding - Transforming documents to embeddings.


Using existing df_complete for BERTopic analysis (titles only).
Processing DataFrame with 15341 rows for BERTopic (using titles only).
Number of non-empty titles for BERTopic: 15341

Initializing BERTopic: Model=all-MiniLM-L6-v2, MinTopicSize=50, TargetTopics=100
Fitting BERTopic model on titles...


Batches:   0%|          | 0/480 [00:00<?, ?it/s]

2025-05-13 17:35:52,299 - BERTopic - Embedding - Completed ✓
2025-05-13 17:35:52,300 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-13 17:35:53,734 - BERTopic - Dimensionality - Completed ✓
2025-05-13 17:35:53,735 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-13 17:35:54,095 - BERTopic - Cluster - Completed ✓
2025-05-13 17:35:54,096 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-13 17:35:54,180 - BERTopic - Representation - Completed ✓
2025-05-13 17:35:54,180 - BERTopic - Topic reduction - Reducing number of topics
2025-05-13 17:35:54,181 - BERTopic - Topic reduction - Reduced number of topics from 40 to 40


BERTopic model fitting complete.

--- BERTopic Results (Titles Only) ---
Number of topics found (after potential reduction): 40
Top topics found:
    Topic  Count                                              Name  \
0      -1   7290               -1_sustainable_digital_using_cancer   
1       0     69             0_perovskite_solar_halide_perovskites   
2       1    182                   1_batteries_battery_ion_lithium   
3       2     64                      2_algae_seaweed_marine_algal   
4       3     93              3_additive_manufacturing_printing_3d   
5       4    278                       4_storage_energy_heat_power   
6       5    554               5_hydrogen_co2_production_catalysts   
7       6    435                 6_waste_sustainable_bio_materials   
8       7   1006                     7_quantum_photonic_spin_light   
9       8    334                 8_gravitational_black_cosmic_dark   
10      9    110          9_energy_buildings_transition_renovation   
11     10     

In [50]:
final_results_df_titles

,id,title,topic_id,topic_label
0,101116741,Digitizing Other Economies: A Comparative Approach,-1,sustainable_digital_learning_cancer
1,101163161,MOLECULAR QUANTUM DYNAMICS IN LOW TEMPERATURE CONDENSED PHASE ASTROCHEMISTRY,-1,sustainable_digital_learning_cancer
2,101160499,Multiscale modelling of aberrant phase transitions in biocondensates,-1,sustainable_digital_learning_cancer
3,101166905,The first comprehensive Atlas of the Milky Way,11,gravitational_black_galaxies_dark
4,101162875,Untapping multiparametric 2D luminescence sensing through MACHine LEarning and Spectral Sorting,-1,sustainable_digital_learning_cancer
...,...,...,...,...
15336,101114220,Deployment of ModelMe Innovation,8,innovation_circular_bioeconomy_regions
15337,101114193,"Cogo - all ride in one app. Cogo gathers electric shared bikes, scooters, cars and mopeds from 250 mobility operators in more than 700 cities, and is now ready to launch the full-service platform.",48,mobility_freight_logistics_urban
15338,101114191,HORIZER,-1,sustainable_digital_learning_cancer
15339,101114035,AI-powered early warning and surveillance system to identify the risks within the food supply chain,22,food_meat_systems_fermentation


In [30]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Configure BERTopic with valid parameters
topic_model = BERTopic(
    # Language model
    language="english",
    
    # Number of topics - "auto" lets the model decide, or specify a number
    nr_topics="auto",  # Or try 20-50 for more control
    
    # Minimum cluster size - higher values create fewer, broader topics
    min_topic_size=30,  # Adjust this: 10-20 for more specific, 50+ for more general
    
    # N-gram range for feature extraction
    n_gram_range=(1, 3),  # Captures single words to 3-word phrases
    
    # Vectorizer settings
    vectorizer_model=CountVectorizer(
        stop_words="english",
        max_features=10000,
        ngram_range=(1, 3)
    ),
    
    # For reproducibility
    seed_topic_list=[0, 1, 2, 3, 4],  # Provide seed topics if needed
    
    # Calculate topic probabilities
    calculate_probabilities=True
)

# Fit the model
docs = df_complete['combined_text'].tolist()
topics, probs = topic_model.fit_transform(docs)

# Add topics to dataframe
df_complete['topic'] = topics
df_complete['topic_probability'] = probs

print(f"Number of topics found: {len(set(topics)) - 1}")  # -1 to exclude outliers
print(f"Number of outliers: {sum(topics == -1)}")

TypeError: can only join an iterable

In [32]:
import pandas as pd
from bertopic import BERTopic
import csv # For saving output with QUOTE_ALL

# --- Assume df_complete is already loaded and prepared in your environment ---
# For example, from your previous CSV parsing script or notebook cell:
#
# df_complete = load_and_fix_cordis_csv(CSV_FILE_PATH) # Or however it was created
#
# If running this as a standalone script and df_complete isn't globally available,
# you would need a way to get it, e.g., load it from a cleaned CSV if you saved one.
# For this example, I'll simulate its creation so the script can run standalone.

# --- SIMULATE df_complete if not already in environment (for standalone script testing) ---
# In your actual Jupyter notebook, you would NOT need this simulation part
# if df_complete is already a defined DataFrame.
if 'df_complete' not in locals() and 'df_complete' not in globals():
    print("Simulating df_complete creation for standalone script testing...")
    # This is a placeholder. Replace with your actual df_complete if running standalone
    # or ensure df_complete is available in your notebook's global scope.
    data_for_simulation = {
        'id': [101116741, 101163161, 101160499, 101116742, 101163162],
        'title': [
            "Digitizing Other Economies: A Comparative Approach",
            "MOLECULAR QUANTUM DYNAMICS IN LOW TEMPERATURE CONDENSED PHASE ASTROCHEMISTRY",
            "Multiscale modelling of aberrant phase transitions in biocondensates",
            "Another Example Title for Science",
            "Tech Innovation Project X"
        ],
        'objective': [
            "How do longstanding, primarily non-industrial, non-capitalist societies adopt and adapt digital technologies... (rest of objective 0)",
            "The James Webb Space Telescope (JWST) has ushered in a new era in observational astrochemistry... (rest of objective 1)",
            "The spatiotemporal organization of the cell material represents one of the great wonders... (rest of objective 2)",
            "This project aims to explore novel scientific frontiers using advanced computational methods and big data analytics to solve complex problems.",
            "Developing cutting-edge technology for societal benefit, focusing on sustainable solutions and user-centric design principles."
        ]
    }
    df_complete = pd.DataFrame(data_for_simulation)
    df_complete['id'] = df_complete['id'].astype(int)
    print(f"Simulated df_complete with {len(df_complete)} rows.")
# --- END SIMULATION ---


# --- BERTopic Configuration (same as before) ---
SAMPLE_SIZE = None  # Set to None to use all data from df_complete, or a number for sampling
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
MIN_TOPIC_SIZE = 10 # Adjusted for potentially smaller datasets if you run on a small df_complete
NR_TOPICS = "auto"


def run_bertopic_on_dataframe(df: pd.DataFrame):
    """
    Runs BERTopic analysis on a pre-loaded DataFrame with 'id', 'title', 'objective'.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        print("ERROR: Input is not a valid or non-empty DataFrame. Exiting.")
        return

    required_cols = ['id', 'title', 'objective']
    if not all(col in df.columns for col in required_cols):
        print(f"ERROR: DataFrame must contain columns: {required_cols}")
        print(f"Available columns: {df.columns.tolist()}")
        return

    print(f"Processing DataFrame with {len(df)} rows for BERTopic.")

    # --- 1. Combine 'title' and 'objective' ---
    print("\nCombining 'title' and 'objective' columns...")
    # Ensure data types are string and handle NaNs before concatenation
    df_working = df.copy() # Work on a copy
    df_working['title'] = df_working['title'].fillna('').astype(str)
    df_working['objective'] = df_working['objective'].fillna('').astype(str)
    df_working['combined_text'] = df_working['title'] + " " + df_working['objective']
    df_working['combined_text'] = df_working['combined_text'].str.strip()

    print("Successfully created 'combined_text' column.")
    if not df_working.empty:
        print(f"Example combined text: '{df_working['combined_text'].iloc[0][:200]}...'")
    
    # Prepare texts and IDs for BERTopic, filtering out empty combined_text
    df_filtered = df_working[df_working['combined_text'] != ''].copy()
    texts_for_bertopic = df_filtered['combined_text'].tolist()
    ids_for_bertopic = df_filtered['id'].tolist() # Original IDs

    if not texts_for_bertopic:
        print("ERROR: No text data available for BERTopic after combining. Check DataFrame content.")
        return
    
    print(f"Number of non-empty documents for BERTopic: {len(texts_for_bertopic)}")

    # Apply sampling if SAMPLE_SIZE is set
    if SAMPLE_SIZE and len(texts_for_bertopic) > SAMPLE_SIZE:
        print(f"Sampling {SAMPLE_SIZE} documents for BERTopic processing...")
        # Sample directly from df_filtered to keep ids and texts aligned
        sampled_df_indices = df_filtered.sample(n=SAMPLE_SIZE, random_state=42).index
        texts_to_process = df_filtered.loc[sampled_df_indices, 'combined_text'].tolist()
        ids_to_process = df_filtered.loc[sampled_df_indices, 'id'].tolist() # Corresponding original IDs
    else:
        print("Processing all available non-empty documents from the DataFrame...")
        texts_to_process = texts_for_bertopic
        ids_to_process = ids_for_bertopic


    # --- 2. Initialize and run BERTopic ---
    print(f"\nInitializing BERTopic with model: {EMBEDDING_MODEL}")
    print(f"Min topic size: {MIN_TOPIC_SIZE}, Nr topics: {NR_TOPICS}")

    topic_model = BERTopic(
        embedding_model=EMBEDDING_MODEL,
        min_topic_size=MIN_TOPIC_SIZE,
        nr_topics=NR_TOPICS,
        verbose=True,
    )

    print("Fitting BERTopic model... This can take a significant amount of time.")
    try:
        topics, probabilities = topic_model.fit_transform(texts_to_process)
        print("BERTopic model fitting complete.")
    except Exception as e:
        print(f"ERROR: BERTopic fitting failed: {e}")
        import traceback
        traceback.print_exc()
        return

    # --- 3. Explore and Save Results ---
    print("\n--- BERTopic Results ---")
    topic_info = topic_model.get_topic_info()
    print("Top topics found:")
    print(topic_info.head(15))

    if -1 in topic_info['Topic'].values:
        outlier_count = topic_info[topic_info['Topic'] == -1]['Count'].iloc[0]
        print(f"\nNumber of outlier documents (Topic -1): {outlier_count}")

    # Create a DataFrame for the processed texts and their topics using original IDs
    results_df = pd.DataFrame({
        'id': ids_to_process, # These are the original IDs of the texts fed to BERTopic
        'topic_id': topics
    })
    
    topic_labels_map = {topic_id: "_".join([word for word, _ in topic_model.get_topic(topic_id)[:3]])
                        for topic_id in set(topics) if topic_model.get_topic(topic_id) is not None}
    if -1 in set(topics): # Handle outliers
        topic_labels_map[-1] = "Outliers"
    results_df['topic_label'] = results_df['topic_id'].map(topic_labels_map)

    # Merge with original titles/objectives from the *filtered and potentially sampled* subset
    # df_filtered contains the 'id', 'title', 'objective', 'combined_text' for all docs that had text
    # We need to merge results_df (which has 'id' and topic info for *processed* docs) back to this.
    
    # Select the subset of df_filtered that corresponds to ids_to_process
    # (This is essentially creating a df from the lists ids_to_process and texts_to_process, then adding original title/objective)
    final_output_df_subset = df_filtered[df_filtered['id'].isin(ids_to_process)][['id', 'title', 'objective', 'combined_text']].copy()
    final_output_df = pd.merge(final_output_df_subset, results_df, on='id', how='left')


    output_csv_path = "cordis_projects_with_bertopics_from_df.csv"
    print(f"\nSaving results with topic assignments to: {output_csv_path}")
    final_output_df.to_csv(output_csv_path, index=False, quoting=csv.QUOTE_ALL)

    model_save_path = "bertopic_cordis_model_from_df"
    print(f"Saving BERTopic model to: {model_save_path}")
    topic_model.save(model_save_path, serialization="safetensors")

    print("\nScript finished successfully!")
    return final_output_df, topic_model # Optionally return results


# --- How to use it ---
if __name__ == "__main__":
    # This __main__ block now assumes df_complete is created or loaded globally
    # In a Jupyter notebook, you would just call:
    # final_df, model = run_bertopic_on_dataframe(df_complete)
    
    if 'df_complete' in locals() or 'df_complete' in globals():
        print("Using existing df_complete for BERTopic analysis.")
        final_results_df, trained_model = run_bertopic_on_dataframe(df_complete)
        if final_results_df is not None:
            print("\nDisplaying head of the final results DataFrame:")
            print(final_results_df.head())
    else:
        print("ERROR: df_complete is not defined. Please ensure it's loaded before running BERTopic analysis.")

2025-05-13 16:36:59,220 - BERTopic - Embedding - Transforming documents to embeddings.


Using existing df_complete for BERTopic analysis.
Processing DataFrame with 15341 rows for BERTopic.

Combining 'title' and 'objective' columns...
Successfully created 'combined_text' column.
Example combined text: 'Digitizing Other Economies: A Comparative Approach How do longstanding, primarily non-industrial, non-capitalist societies adopt and adapt digital technologies in their daily practices and systems of ...'
Number of non-empty documents for BERTopic: 15341
Processing all available non-empty documents from the DataFrame...

Initializing BERTopic with model: all-MiniLM-L6-v2
Min topic size: 10, Nr topics: auto
Fitting BERTopic model... This can take a significant amount of time.


Batches:   0%|          | 0/480 [00:00<?, ?it/s]

2025-05-13 16:38:05,885 - BERTopic - Embedding - Completed ✓
2025-05-13 16:38:05,885 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-13 16:38:14,715 - BERTopic - Dimensionality - Completed ✓
2025-05-13 16:38:14,716 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-13 16:38:14,959 - BERTopic - Cluster - Completed ✓
2025-05-13 16:38:14,959 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-13 16:38:16,134 - BERTopic - Representation - Completed ✓
2025-05-13 16:38:16,137 - BERTopic - Topic reduction - Reducing number of topics
2025-05-13 16:38:17,241 - BERTopic - Topic reduction - Reduced number of topics from 243 to 12


BERTopic model fitting complete.

--- BERTopic Results ---
Top topics found:
    Topic  Count                             Name  \
0      -1   5886                 -1_the_and_of_to   
1       0   9138                  0_the_and_of_to   
2       1     84         1_of_the_theory_geometry   
3       2     62               2_of_the_to_theory   
4       3     46             3_the_vr_and_virtual   
5       4     31              4_the_causal_of_and   
6       5     21          5_kidney_ckd_disease_of   
7       6     19    6_the_reproductive_and_embryo   
8       7     15  7_hearing_auditory_the_cochlear   
9       8     15           8_dna_storage_data_and   
10      9     13         9_circadian_clock_the_in   
11     10     11   10_equations_the_dispersive_of   

                                                                  Representation  \
0                                [the, and, of, to, in, will, for, is, with, on]   
1                                [the, and, of, to, in, will, for

In [33]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Configure BERTopic without problematic parameters
topic_model = BERTopic(
    language="english",
    min_topic_size=30,  # Adjust this: 10-20 for more specific, 50+ for more general
    nr_topics="auto",  # Or try 20-50 for more control
    n_gram_range=(1, 3),
    vectorizer_model=CountVectorizer(
        stop_words="english",
        max_features=10000,
        ngram_range=(1, 3)
    ),
    calculate_probabilities=True
)

# Fit the model
docs = df_complete['combined_text'].tolist()
topics, probs = topic_model.fit_transform(docs)

# Add topics to dataframe
df_complete['topic'] = topics
df_complete['topic_probability'] = probs

print(f"Number of topics found: {len(set(topics)) - 1}")  # -1 to exclude outliers
print(f"Number of outliers: {sum(topics == -1)}")

ValueError: Expected a 1D array, got an array with shape (15341, 44)

In [34]:
from bertopic import BERTopic
import pandas as pd

# Configure BERTopic without probability calculation
topic_model = BERTopic(
    language="english",
    min_topic_size=30,  
    nr_topics="auto",
    n_gram_range=(1, 3),
    calculate_probabilities=False  # Don't calculate probabilities
)

# Fit the model
docs = df_complete['combined_text'].tolist()
topics, probs = topic_model.fit_transform(docs)

# Add topics to dataframe
df_complete['topic'] = topics

print(f"Number of topics found: {len(set(topics)) - 1}")
print(f"Number of outliers: {sum(topics == -1)}")

Number of topics found: 45


TypeError: 'bool' object is not iterable

In [36]:
# Properly count topics and outliers
unique_topics = set(topics)
num_topics = len(unique_topics) - (1 if -1 in unique_topics else 0)
num_outliers = len([t for t in topics if t == -1])

print(f"Number of topics found: {num_topics}")
print(f"Number of outliers: {num_outliers}")
print(f"Total documents: {len(topics)}")

Number of topics found: 45
Number of outliers: 6648
Total documents: 15341


In [37]:
# Get topic information
topic_info = topic_model.get_topic_info()
print("\nTopic overview:")
print(topic_info.head(10))

# Get top words for each topic
print("\nTop words per topic:")
for topic_num in topic_info['Topic'].values[:10]:
    if topic_num == -1:  # Skip outliers
        continue
    topic_words = topic_model.get_topic(topic_num)
    words = [word[0] for word in topic_words[:5]]
    count = len(df_complete[df_complete['topic'] == topic_num])
    print(f"Topic {topic_num} ({count} docs): {', '.join(words)}")


Topic overview:
   Topic  Count                  Name  \
0     -1   6648      -1_and_the_of_to   
1      0   3068       0_and_the_of_to   
2      1    488   1_quantum_the_of_to   
3      2    305       2_the_of_in_and   
4      3    298       3_and_the_of_in   
5      4    283       4_the_and_of_to   
6      5    267       5_the_of_and_in   
7      6    247       6_the_of_and_to   
8      7    241  7_cancer_cells_of_to   
9      8    237       8_the_of_and_to   

                                                  Representation  \
0                [and, the, of, to, in, for, will, is, with, on]   
1            [and, the, of, to, in, for, will, with, of the, be]   
2   [quantum, the, of, to, and, in, will, for, with, of quantum]   
3              [the, of, in, and, to, will, is, that, brain, ad]   
4       [and, the, of, in, to, on, social, political, how, will]   
5            [the, and, of, to, climate, in, will, for, ice, on]   
6        [the, of, and, in, of the, to, history, in the

In [38]:
from sklearn.feature_extraction.text import CountVectorizer

# Inside your run_bertopic_on_dataframe function, before initializing BERTopic:

# Define a CountVectorizer with English stopwords
vectorizer = CountVectorizer(stop_words="english")

# Then, when initializing BERTopic:
topic_model = BERTopic(
    embedding_model=EMBEDDING_MODEL,
    vectorizer_model=vectorizer,  # ADD THIS LINE
    min_topic_size=MIN_TOPIC_SIZE,
    nr_topics=NR_TOPICS,
    verbose=True,
)